Gradio Example

In [1]:
import os
from dotenv import load_dotenv

from openai import OpenAI
import gradio as gr



In [2]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

In [3]:
openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)


In [4]:
system_message = "You are a helpfull assistant"

def message_gpt(prompt):
    messages  = [{"role": "system" , "content":system_message},{"role": "user","content":prompt}]
    response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
    return response.choices[0].message.content


In [5]:
message_gpt("What is today's date?")

"Today's date is April 27, 2024."

User interface time

In [10]:
# here's a simple function

def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()

In [11]:
shout("hello")

Shout has been called with input hello


'HELLO'

In [12]:
gr.Interface(fn=shout,inputs="textbox",outputs="textbox",flagging_mode="never").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input Hi Avinash


In [ ]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(share=True)

In [15]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input io


### Gradio Example for User Login

In [16]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True, auth=("shelke", "avinash"))

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input Hey Babu


### Feedback Form UI

In [6]:
import gradio as gr

def submit_feedback(name, rating, category, comments):
    # In a real app, you might save this to a database or CSV
    summary = f"""
    ### Feedback Received!
    **User:** {name}
    **Rating:** {rating}/10
    **Category:** {category}
    **Comments:** {comments}
    """
    return summary

# Define the UI components
with gr.Blocks(title="Course Feedback Form") as demo:
    gr.Markdown("# 📋 User Feedback Form")
    gr.Markdown("Please let us know how we're doing!")
    
    with gr.Row():
        name_input = gr.Textbox(label="Your Name", placeholder="Enter your name here...")
        rating_input = gr.Slider(minimum=1, maximum=10, step=1, label="Overall Rating", value=5)
    
    category_input = gr.Radio(
        choices=["Technical Content", "UI/UX", "Performance", "Other"], 
        label="Feedback Category"
    )
    
    comment_input = gr.Textbox(label="Detailed Comments", lines=4)
    
    submit_btn = gr.Button("Submit Feedback", variant="primary")
    
    output_display = gr.Markdown(label="Submission Status")

    # Link the button to the function
    submit_btn.click(
        fn=submit_feedback,
        inputs=[name_input, rating_input, category_input, comment_input],
        outputs=output_display
    )

    
    demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### Building Company Product Catalog

In [7]:
from scraper import fetch_website_contents

In [11]:

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [9]:
# Again this is typical Experimental mindset - I'm changing the global variable we used above:

system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short products catalog with price and description , manufacturer product category.
Respond in markdown without code blocks.
"""

In [13]:
def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = openai.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [12]:

def stream_ollama(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = ollama.chat.completions.create(
        model='llama3.2:1b',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [14]:
def stream_catalog(company_name, url, model):
    yield ""
    prompt = f"Please generate a company product catalog for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    if model=="GPT":
        result = stream_gpt(prompt)
    elif model=="Ollama":
        result = stream_ollama(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [16]:
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["GPT", "Ollama"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_catalog,
    title="Product Catalog Generator", 
    inputs=[name_input, url_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Hyundai", "https://www.hyundai.com/", "GPT"],
            ["Tata Motors", "https://cars.tatamotors.com/", "Ollama"]
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
